# Fine-tune Khmer XLS-R on the prepared dataset

Run the cells in order. The fine-tuning script now rebuilds and validates a clean CTC tokenizer before preparing labels. The smoke-training cell uses only a few examples. Run the full-training cell only after the smoke run completes successfully. The test split is kept out of training. The corrected checkpoint is saved as khmer_xlsr_ddd_1000_clean so the previous model remains available for comparison.

In [22]:
import sys
import torch

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Python: /usr/bin/python3
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [37]:
!pip install -q -r /content/requirements-asr.txt

In [38]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount("/content/drive")

drive_data = Path("/content/drive/MyDrive/khmer_asr/voice_1000")
colab_data = Path("/content/data/voice_1000")
colab_data.parent.mkdir(parents=True, exist_ok=True)
if not colab_data.exists():
    shutil.copytree(drive_data, colab_data)
print("Manifest directory:", colab_data)
print("Files:", sorted(path.name for path in colab_data.iterdir()))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Manifest directory: /content/data/voice_1000
Files: ['baseline_smoke_test.json', 'baseline_test.json', 'manifest_summary.json', 'manifest_validation.json', 'test.csv', 'train.csv', 'validation.csv']


Upload the individual files from scripts/ and khmer_transliteration/ to /content with the Colab extension. The next cell places the uploaded files in the package folders expected by Python. Required shared modules are asr_audio.py, asr_data.py, asr_inference.py, asr_text.py, paths.py, and asr_metrics.py.

In [39]:
from pathlib import Path
import shutil

Path("/content/scripts").mkdir(parents=True, exist_ok=True)
Path("/content/khmer_transliteration").mkdir(parents=True, exist_ok=True)

file_moves = {
    "/content/finetune_asr.py": "/content/scripts/finetune_asr.py",
    "/content/run_asr_baseline.py": "/content/scripts/run_asr_baseline.py",
    "/content/asr_text.py": "/content/khmer_transliteration/asr_text.py",
    "/content/asr_audio.py": "/content/khmer_transliteration/asr_audio.py",
    "/content/asr_data.py": "/content/khmer_transliteration/asr_data.py",
    "/content/asr_inference.py": "/content/khmer_transliteration/asr_inference.py",
    "/content/asr_metrics.py": "/content/khmer_transliteration/asr_metrics.py",
    "/content/paths.py": "/content/khmer_transliteration/paths.py",
    "/content/__init__.py": "/content/khmer_transliteration/__init__.py",
}
for source, target in file_moves.items():
    if Path(source).exists():
        shutil.copy2(source, target)
        print("Ready:", target)
    elif not Path(target).exists():
        print("Missing; upload this file:", source)

## 1. Materialize the selected audio

This replays the manifest order on Hugging Face and stores the 793 train, 81 validation, and 126 test WAV files in the temporary Colab runtime. The test files are stored only for later evaluation.

In [40]:
!python /content/scripts/finetune_asr.py --stage materialize --manifest-dir /content/data/voice_1000 --materialized-dir /content/data/voice_1000/materialized_training

Resolving data files: 100% 4447/4447 [00:00<00:00, 24988.44it/s]
Resolving data files: 100% 4447/4447 [00:00<00:00, 26905.49it/s]
Materialized audio to: /content/data/voice_1000/materialized_training
Counts: {'train': 793, 'validation': 81, 'test': 126}


In [41]:
import json
from pathlib import Path

summary_path = Path("/content/data/voice_1000/materialized_training/materialization_summary.json")
print(json.loads(summary_path.read_text(encoding="utf-8")))

{'dataset': 'Digital-Divide-Data/khmer-speech-dataset', 'source_split': 'train', 'seed': 42, 'shuffle_buffer': 5000, 'counts': {'train': 793, 'validation': 81, 'test': 126}, 'target_sample_rate': 16000, 'note': 'Test files are materialized for evaluation but excluded from training.'}


## 2. Smoke training

This runs five optimizer steps on at most 8 training and 4 validation examples. It checks the processor, CTC labels, data collator, GPU, and Trainer before spending time on the full run.

In [45]:
from pathlib import Path
required_files = [
    Path("/content/scripts/finetune_asr.py"),
    Path("/content/khmer_transliteration/asr_audio.py"),
    Path("/content/khmer_transliteration/asr_data.py"),
    Path("/content/khmer_transliteration/asr_inference.py"),
    Path("/content/khmer_transliteration/asr_text.py"),
]
missing = [str(path) for path in required_files if not path.exists()]
if missing:
    raise FileNotFoundError(f"Upload these files first: {missing}")
print("ASR project files are ready.")

Ready: True


In [46]:
!python /content/scripts/finetune_asr.py --stage smoke --manifest-dir /content/data/voice_1000 --materialized-dir /content/data/voice_1000/materialized_training --output-dir /content/models/khmer_xlsr_ddd_smoke

Loading weights: 100% 424/424 [00:00<00:00, 4265.49it/s]
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 74, 'bos_token_id': 73}.
{'loss': '5.033', 'grad_norm': '20.38', 'learning_rate': '1e-05', 'epoch': '0.5'}
{'loss': '4.784', 'grad_norm': 'nan', 'learning_rate': '8e-06', 'epoch': '1'}
{'loss': '4.994', 'grad_norm': '14.17', 'learning_rate': '6e-06', 'epoch': '1.5'}
{'loss': '4.386', 'grad_norm': '11.68', 'learning_rate': '4e-06', 'epoch': '2'}
{'loss': '4.142', 'grad_norm': '21.46', 'learning_rate': '2e-06', 'epoch': '2.5'}
100% 5/5 [00:06<00:00,  1.12s/it]
  0% 0/2 [00:00<?, ?it/s]
                                 
{'eval_loss': '1.283', 'eval_cer': '0.3187', 'eval_wer': '4.143', 'eval_sentence_accuracy': '0', 'eval_runtime': '0.4642', 'eval_samples_per_second': '8.616', 'eval_s

In [47]:
import json
from pathlib import Path

smoke_summary = Path("/content/models/khmer_xlsr_ddd_smoke/smoke_training_summary.json")
print(json.loads(smoke_summary.read_text(encoding="utf-8")))

{'stage': 'smoke', 'model': 'vitouphy/wav2vec2-xls-r-300m-khmer', 'train_rows': 8, 'validation_rows': 4, 'train_metrics': {'train_runtime': 7.144, 'train_samples_per_second': 2.8, 'train_steps_per_second': 0.7, 'total_flos': 6896918494224000.0, 'train_loss': 4.667923545837402, 'epoch': 2.5}, 'validation_metrics': {'eval_loss': 1.2833492755889893, 'eval_cer': 0.31875, 'eval_wer': 4.142857142857143, 'eval_sentence_accuracy': 0.0, 'eval_runtime': 0.5891, 'eval_samples_per_second': 6.79, 'eval_steps_per_second': 3.395, 'epoch': 2.5}, 'note': 'The test split was not used during fine-tuning.'}


## 3. Full fine-tuning

Run this only when smoke training succeeds. It uses all train and validation rows for three epochs; the test split remains untouched.

In [48]:
!python /content/scripts/finetune_asr.py --stage full --manifest-dir /content/data/voice_1000 --materialized-dir /content/data/voice_1000/materialized_training --output-dir /content/models/khmer_xlsr_ddd_1000_clean

Loading weights: 100% 424/424 [00:00<00:00, 4384.58it/s]
Skipping unsupported TrainingArguments options: ['group_by_length']
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 74, 'bos_token_id': 73}.
{'loss': '5.431', 'grad_norm': '14.91', 'learning_rate': '1e-05', 'epoch': '0.005038'}
{'loss': '4.764', 'grad_norm': '15.96', 'learning_rate': '9.983e-06', 'epoch': '0.01008'}
{'loss': '4.422', 'grad_norm': '10.05', 'learning_rate': '9.966e-06', 'epoch': '0.01511'}
{'loss': '4.281', 'grad_norm': '9.214', 'learning_rate': '9.95e-06', 'epoch': '0.02015'}
{'loss': '3.851', 'grad_norm': '17.26', 'learning_rate': '9.933e-06', 'epoch': '0.02519'}
{'loss': '4.576', 'grad_norm': '9.286', 'learning_rate': '9.916e-06', 'epoch': '0.03023'}
{'loss': '3.762', 'grad_norm': '8.206', 'learning_rate': '9.

In [49]:
from pathlib import Path
import shutil

local_model = Path("/content/models/khmer_xlsr_ddd_1000_clean")
drive_model = Path("/content/drive/MyDrive/khmer_asr/models/khmer_xlsr_ddd_1000_clean")
drive_model.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(local_model, drive_model, dirs_exist_ok=True)
print("Saved model to:", drive_model)

Saved model to: /content/drive/MyDrive/khmer_asr/models/khmer_xlsr_ddd_1000_clean


## 4. Evaluate the fine-tuned model on the untouched test split

In [50]:
!python /content/scripts/run_asr_baseline.py --model /content/models/khmer_xlsr_ddd_1000_clean --manifest-dir /content/data/voice_1000 --split test --max-samples 0 --output /content/data/voice_1000/finetuned_test_clean.json

Resolving data files: 100% 4447/4447 [00:00<00:00, 26945.99it/s]
Resolving data files: 100% 4447/4447 [00:00<00:00, 17097.32it/s]
Loading weights: 100% 424/424 [00:00<00:00, 826.33it/s]
Model: /content/models/khmer_xlsr_ddd_1000_clean
Device: cuda
Successful samples: 126
Errors: 0
Resampled to 16 kHz: 0
Metrics: {'cer': 0.21844481297217616, 'wer': 1.0144578313253012, 'sentence_accuracy': 0.0, 'mean_latency_seconds': 0.09527644272222663, 'median_latency_seconds': 0.08455128950026847, 'p95_latency_seconds': 0.14036579399908078, 'mean_real_time_factor': 0.01390838311227837}
Saved report to: /content/data/voice_1000/finetuned_test_clean.json


In [51]:
from pathlib import Path
import json
import shutil

report_path = Path("/content/data/voice_1000/finetuned_test_clean.json")
report = json.loads(report_path.read_text(encoding="utf-8"))
print("Model:", report["model"])
print("Metrics:", report["metrics"])

drive_report = Path("/content/drive/MyDrive/khmer_asr/voice_1000/finetuned_test_clean.json")
shutil.copy2(report_path, drive_report)
print("Saved report to:", drive_report)

Model: /content/models/khmer_xlsr_ddd_1000_clean
Metrics: {'cer': 0.21844481297217616, 'wer': 1.0144578313253012, 'sentence_accuracy': 0.0, 'mean_latency_seconds': 0.09527644272222663, 'median_latency_seconds': 0.08455128950026847, 'p95_latency_seconds': 0.14036579399908078, 'mean_real_time_factor': 0.01390838311227837}
Saved report to: /content/drive/MyDrive/khmer_asr/voice_1000/finetuned_test_clean.json


In [3]:
from pathlib import Path
import shutil

Path("/content/scripts").mkdir(parents=True, exist_ok=True)
Path("/content/khmer_transliteration").mkdir(parents=True, exist_ok=True)

file_moves = {
    "/content/finetune_asr.py":
        "/content/scripts/finetune_asr.py",

    "/content/run_asr_baseline.py":
        "/content/scripts/run_asr_baseline.py",

    "/content/asr_inference.py":
        "/content/khmer_transliteration/asr_inference.py",

    "/content/asr_audio.py":
        "/content/khmer_transliteration/asr_audio.py",

    "/content/asr_data.py":
        "/content/khmer_transliteration/asr_data.py",

    "/content/asr_text.py":
        "/content/khmer_transliteration/asr_text.py",

    "/content/asr_metrics.py":
        "/content/khmer_transliteration/asr_metrics.py",

    "/content/paths.py":
        "/content/khmer_transliteration/paths.py",

    "/content/__init__.py":
        "/content/khmer_transliteration/__init__.py",
}

for source, target in file_moves.items():
    source_path = Path(source)
    target_path = Path(target)

    if source_path.exists():
        target_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source_path, target_path)
        print("Copied:", target)
    else:
        print("Missing upload:", source)

Copied: /content/scripts/finetune_asr.py
Copied: /content/scripts/run_asr_baseline.py
Copied: /content/khmer_transliteration/asr_inference.py
Copied: /content/khmer_transliteration/asr_audio.py
Copied: /content/khmer_transliteration/asr_data.py
Copied: /content/khmer_transliteration/asr_text.py
Copied: /content/khmer_transliteration/asr_metrics.py
Copied: /content/khmer_transliteration/paths.py
Copied: /content/khmer_transliteration/__init__.py


In [4]:
from pathlib import Path

required_files = [
    "/content/scripts/finetune_asr.py",
    "/content/scripts/run_asr_baseline.py",
    "/content/khmer_transliteration/asr_inference.py",
    "/content/khmer_transliteration/asr_audio.py",
    "/content/khmer_transliteration/asr_data.py",
    "/content/khmer_transliteration/asr_text.py",
    "/content/khmer_transliteration/asr_metrics.py",
    "/content/khmer_transliteration/paths.py",
    "/content/khmer_transliteration/__init__.py",
]

for file_path in required_files:
    print("OK    " if Path(file_path).exists() else "MISSING", file_path)

OK     /content/scripts/finetune_asr.py
OK     /content/scripts/run_asr_baseline.py
OK     /content/khmer_transliteration/asr_inference.py
OK     /content/khmer_transliteration/asr_audio.py
OK     /content/khmer_transliteration/asr_data.py
OK     /content/khmer_transliteration/asr_text.py
OK     /content/khmer_transliteration/asr_metrics.py
OK     /content/khmer_transliteration/paths.py
OK     /content/khmer_transliteration/__init__.py


In [5]:
!pip install -q -r /content/requirements-asr.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 49.2 MB/s eta 0:00:00a 0:00:01


In [6]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [7]:
from pathlib import Path
import shutil

drive_data = Path("/content/drive/MyDrive/khmer_asr/voice_1000")
colab_data = Path("/content/data/voice_1000")

colab_data.parent.mkdir(parents=True, exist_ok=True)

if not colab_data.exists():
    shutil.copytree(drive_data, colab_data)

print("Dataset location:", colab_data)
print(sorted(file.name for file in colab_data.iterdir()))

Dataset location: /content/data/voice_1000
['baseline_test.json', 'finetuned_test.json', 'manifest_summary.json', 'manifest_validation.json', 'test.csv', 'train.csv', 'validation.csv']


In [8]:
!python /content/scripts/finetune_asr.py \
    --stage materialize \
    --manifest-dir /content/data/voice_1000 \
    --materialized-dir /content/data/voice_1000/materialized_training

README.md: 100% 4.28k/4.28k [00:00<00:00, 5.30MB/s]
Resolving data files: 100% 4447/4447 [00:00<00:00, 25147.22it/s]
Resolving data files: 100% 4447/4447 [00:00<00:00, 27309.27it/s]
Materialized audio to: /content/data/voice_1000/materialized_training
Counts: {'train': 793, 'validation': 81, 'test': 126}


In [9]:
!python /content/scripts/finetune_asr.py \
    --stage validate-tokenizer \
    --model vitouphy/wav2vec2-xls-r-300m-khmer

config.json: 100% 2.07k/2.07k [00:00<00:00, 5.16MB/s]

model.safetensors: downloading bytes:  14% 172M/1.26G [00:01<00:04, 222MB/s, 13.1MB/s  ]  
model.safetensors: downloading bytes:  94% 1.18G/1.26G [00:05<00:00, 241MB/s, 89.3MB/s  ]
model.safetensors: downloading bytes:  95% 1.19G/1.26G [00:18<00:00, 241MB/s, 89.6MB/s  ] ]
model.safetensors: reconstructing file:  74% 939M/1.26G [00:18<00:06, 49.2MB/s, 12.7MB/s  ]
model.safetensors: downloading bytes: 100% 1.19G/1.19G [00:19<00:00, 60.1MB/s, 89.6MB/s  ] ]
model.safetensors: reconstructing file: 100% 1.26G/1.26G [00:19<00:00, 63.6MB/s, 83.6MB/s  ]
Loading weights: 100% 424/424 [00:00<00:00, 11157.51it/s]
preprocessor_config.json: 100% 262/262 [00:00<00:00, 1.22MB/s]
vocab.json: 100% 795/795 [00:00<00:00, 4.17MB/s]
{
  "tokenizer_length": 75,
  "base_vocab_size": 73,
  "delimiter_token": "|",
  "delimiter_token_id": 0,
  "unk_token_id": 71,
  "pad_token_id": 72,
  "labels_add_special_tokens": false,
  "model_vocab_size": 75,
  "model_o

In [10]:
!python /content/scripts/finetune_asr.py \
    --stage smoke \
    --manifest-dir /content/data/voice_1000 \
    --materialized-dir /content/data/voice_1000/materialized_training \
    --output-dir /content/models/khmer_xlsr_ddd_1000_clean_smoke

Loading weights: 100% 424/424 [00:00<00:00, 5906.29it/s]
Tokenizer/model validation:
{
  "tokenizer_length": 75,
  "base_vocab_size": 73,
  "delimiter_token": "|",
  "delimiter_token_id": 0,
  "unk_token_id": 71,
  "pad_token_id": 72,
  "labels_add_special_tokens": false,
  "model_vocab_size": 75,
  "model_output_classes": 75,
  "compatible": true
}
Generating train split: 793 examples [00:00, 201825.43 examples/s]
Generating validation split: 81 examples [00:00, 56266.75 examples/s]
Preparing audio and CTC labels: 100% 793/793 [00:03<00:00, 201.44 examples/s]
Preparing audio and CTC labels: 100% 81/81 [00:00<00:00, 377.12 examples/s]
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 74, 'bos_token_id': 73}.
{'loss': '5.033', 'grad_norm': '20.38', 'learning_rate': '1e-05', 'epoch': '0.

In [11]:
!python /content/scripts/finetune_asr.py \
    --stage full \
    --manifest-dir /content/data/voice_1000 \
    --materialized-dir /content/data/voice_1000/materialized_training \
    --output-dir /content/models/khmer_xlsr_ddd_1000_clean

Loading weights: 100% 424/424 [00:00<00:00, 5348.75it/s]
Tokenizer/model validation:
{
  "tokenizer_length": 75,
  "base_vocab_size": 73,
  "delimiter_token": "|",
  "delimiter_token_id": 0,
  "unk_token_id": 71,
  "pad_token_id": 72,
  "labels_add_special_tokens": false,
  "model_vocab_size": 75,
  "model_output_classes": 75,
  "compatible": true
}
Skipping unsupported TrainingArguments options: ['group_by_length']
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 74, 'bos_token_id': 73}.
{'loss': '5.431', 'grad_norm': '14.91', 'learning_rate': '1e-05', 'epoch': '0.005038'}
{'loss': '4.764', 'grad_norm': '15.74', 'learning_rate': '9.983e-06', 'epoch': '0.01008'}
{'loss': '4.422', 'grad_norm': '10.05', 'learning_rate': '9.966e-06', 'epoch': '0.01511'}
{'loss': '4.28', 'grad_norm': '9.1

In [1]:
from pathlib import Path
import shutil

local_model = Path("/content/models/khmer_xlsr_ddd_1000_clean")
drive_model = Path(
    "/content/drive/MyDrive/khmer_asr/models/khmer_xlsr_ddd_1000_clean"
)

drive_model.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(local_model, drive_model, dirs_exist_ok=True)

print("Saved model to:", drive_model)

Saved model to: /content/drive/MyDrive/khmer_asr/models/khmer_xlsr_ddd_1000_clean


In [ ]:
!python /content/scripts/run_asr_baseline.py \
  --model /content/models/khmer_xlsr_ddd_1000_expanded \
  --manifest-dir /content/data/voice_1000 \
  --split test \
  --max-samples 0 \
  --decoder greedy_ctc \
  --output /content/data/voice_1000/finetuned_test_expanded.json

Loaded 126/126 audio files from cache: /content/data/voice_1000/audio_cache
Loading ASR model on cuda: /content/models/khmer_xlsr_ddd_1000_expanded
Loading weights: 100% 424/424 [00:00<00:00, 4971.67it/s]
Running inference on 126 samples...
Inference progress: 1/126
Inference progress: 2/126
Inference progress: 3/126
Inference progress: 4/126
Inference progress: 5/126
Inference progress: 6/126
Inference progress: 7/126
Inference progress: 8/126
Inference progress: 9/126
Inference progress: 10/126
Inference progress: 11/126
Inference progress: 12/126
Inference progress: 13/126
Inference progress: 14/126
Inference progress: 15/126
Inference progress: 16/126
Inference progress: 17/126
Inference progress: 18/126
Inference progress: 19/126
Inference progress: 20/126
Inference progress: 21/126
Inference progress: 22/126
Inference progress: 23/126
Inference progress: 24/126
Inference progress: 25/126
Inference progress: 26/126
Inference progress: 27/126
Inference progress: 28/126
Inference pr

In [ ]:
!python /content/scripts/finetune_asr.py \
  --stage full \
  --manifest-dir /content/data/voice_5000 \
  --materialized-dir /content/data/voice_5000/materialized_training \
  --model /content/models/khmer_xlsr_ddd_1000_expanded \
  --output-dir /content/models/khmer_xlsr_ddd_5000_expanded

Loading weights: 100% 424/424 [00:00<00:00, 4551.06it/s]
Tokenizer/model validation:
{
  "tokenizer_length": 166,
  "base_vocab_size": 164,
  "delimiter_token": "|",
  "delimiter_token_id": 0,
  "unk_token_id": 71,
  "pad_token_id": 72,
  "labels_add_special_tokens": false,
  "model_vocab_size": 166,
  "model_output_classes": 166,
  "compatible": true
}
Vocabulary expansion: added 25 train/validation characters
Generating train split: 3570 examples [00:00, 250061.21 examples/s]
Generating validation split: 681 examples [00:00, 248765.11 examples/s]
Preparing audio and CTC labels: 100% 3570/3570 [01:12<00:00, 49.50 examples/s] 
Preparing audio and CTC labels: 100% 681/681 [00:03<00:00, 177.91 examples/s]
Skipping unsupported TrainingArguments options: ['group_by_length']
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!mkdir -p /content/drive/MyDrive/khmer_asr/models/khmer_xlsr_ddd_5000_expanded
!cp -r /content/models/khmer_xlsr_ddd_5000_expanded/. \
    /content/drive/MyDrive/khmer_asr/models/khmer_xlsr_ddd_5000_expanded/